In [1]:
# 9_group_averages.ipynb
#
# Computes group-level baselines for each (unit_id, employment group) from the
# synthetic population parquet.  Outputs two CSVs consumed by step 8:
#
#   group_baselines.csv        — one row per (unit_id, group) with continuous
#                                 means and categorical modes (same schema as
#                                 the cluster CSV for easy comparison).
#
#   group_distributions.csv    — long format: one row per (unit_id, group,
#                                 variable, category) with population counts
#                                 and percentages.
#
# These allow step 8 (label_clusters) to tell the LLM how each cluster
# differs from its group average, producing more insightful descriptions.

import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_pipeline.config_paths as _cp
importlib.reload(_cp)
from data_pipeline.config_paths import DATA_FOLDER, USE_FOUR_LA_SUBSET, FOUR_LA_CODES
import data_pipeline.config_variables as _cv
import data_pipeline.config_cluster   as _cc
_cv.reload_config_variables()
importlib.reload(_cc)

import pandas as pd
import numpy  as np
from pathlib import Path
from tqdm    import tqdm

import data_pipeline.helpers.cluster as cf
importlib.reload(cf)

from data_pipeline.config_variables import (
    SUMMARY_VARS, VARIABLE_MAP,
    CATEGORICAL_VARS, CATEGORY_MAPS, CONTINUOUS_VARS,
)
from data_pipeline.config_cluster import WAVE, GROUPS

# ── Config ────────────────────────────────────────────────────────────────────
CLUSTER_LEVEL    = "LA"
SYNPOP_PARQUET   = f"../{DATA_FOLDER}/6_synthetic_population/synthetic_population.parquet"
CLUSTER_CSV      = Path(f"../{DATA_FOLDER}/7_cluster_local_level/LA_london_clusters.csv")
OUTPUT_DIR       = Path(f"../{DATA_FOLDER}/9_group_averages")
GEO_CSV          = Path(f"../{DATA_FOLDER}/0_raw/admin_geography_mappings.csv")
# Same geography scope as 7_cluster_local_level.ipynb (config_paths.USE_FOUR_LA_SUBSET)
UNIT_FILTER = list(FOUR_LA_CODES) if USE_FOUR_LA_SUBSET else "london"

LEVEL_COL = "ladcd"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BASELINES_CSV     = OUTPUT_DIR / "group_baselines.csv"
DISTRIBUTIONS_CSV = OUTPUT_DIR / "group_distributions.csv"

if not os.path.exists(SYNPOP_PARQUET):
    raise FileNotFoundError(f"{SYNPOP_PARQUET} not found — run 6_synthetic_population first.")

# ── LA name lookup ────────────────────────────────────────────────────────────
_la_name_map = {}
if GEO_CSV.exists():
    _geo = pd.read_csv(GEO_CSV, encoding='latin-1', usecols=['ladcd', 'ladnm']).drop_duplicates('ladcd')
    _la_name_map = _geo.set_index('ladcd')['ladnm'].to_dict()

# ── Jbstat group assignment (same logic as step 6) ────────────────────────────
import pyarrow.parquet as pq
parquet_cols = set(pq.read_schema(SYNPOP_PARQUET).names)

_JBSTAT_LOOKUP = {
    "Employed":   f"{WAVE}_jbstat_1",
    "Unemployed": f"{WAVE}_jbstat_3",
    "Retired":    f"{WAVE}_jbstat_4",
    "On leave":   f"{WAVE}_jbstat_5",
    "Student":    f"{WAVE}_jbstat_7",
    "Inactive":   f"{WAVE}_jbstat_8",
}
GROUP_COL = {
    gname: (
        _JBSTAT_LOOKUP[gname]
        if _JBSTAT_LOOKUP.get(gname) in parquet_cols
        else (None if _JBSTAT_LOOKUP.get(gname) is None else "MISSING")
    )
    for gname in GROUPS
}

# ── Determine units from the cluster CSV (so we only process what was clustered)
df_clusters = pd.read_csv(CLUSTER_CSV)
unit_ids = sorted(df_clusters['unit_id'].unique().tolist())
if UNIT_FILTER is None:
    pass
elif UNIT_FILTER == "london":
    unit_ids = [u for u in unit_ids if str(u).startswith("E09")]
elif isinstance(UNIT_FILTER, list):
    _allow = set(UNIT_FILTER)
    unit_ids = [u for u in unit_ids if u in _allow]
else:
    raise ValueError("UNIT_FILTER must be None, 'london', or a list of LA codes")
if not unit_ids:
    raise ValueError(
        "No units after UNIT_FILTER — cluster CSV may not match config_paths "
        "(re-run 7_cluster_local_level with the same USE_FOUR_LA_SUBSET)."
    )
print(f"Computing group averages for {len(unit_ids)} units [USE_FOUR_LA_SUBSET={USE_FOUR_LA_SUBSET}]")

# ── Main loop ─────────────────────────────────────────────────────────────────
baseline_rows = []
dist_rows     = []

for unit_id in tqdm(unit_ids, desc="Group averages"):
    df_unit = pd.read_parquet(
        SYNPOP_PARQUET,
        filters=[(LEVEL_COL, '==', unit_id)]
    )
    if df_unit.empty:
        continue

    la_name = _la_name_map.get(unit_id, unit_id)
    assigned = pd.Series(False, index=df_unit.index)

    for gname in GROUPS:
        col = GROUP_COL.get(gname)
        if col == "MISSING":
            continue
        if col is not None and col in df_unit.columns:
            mask = df_unit[col] == 1.0
        elif col is None:
            mask = ~assigned
        else:
            continue

        group_df = df_unit[mask]
        assigned |= mask
        if group_df.empty:
            continue

        # ── Baseline row (same schema as cluster CSV) ─────────────────────
        row = cf.build_dna_row(
            f"{gname} (baseline)", group_df, WAVE,
            SUMMARY_VARS, VARIABLE_MAP, CATEGORICAL_VARS, CATEGORY_MAPS,
            continuous_vars=CONTINUOUS_VARS,
        )
        row['unit_id'] = unit_id
        row['la_name'] = la_name
        row['group']   = gname
        baseline_rows.append(row)

        # ── Full categorical distributions ────────────────────────────────
        for base_code in SUMMARY_VARS:
            if base_code not in CATEGORICAL_VARS:
                continue
            cat_map = CATEGORY_MAPS.get(base_code)
            if not cat_map:
                continue
            col_name = f"{WAVE}_{base_code}"
            if col_name not in group_df.columns:
                continue

            label  = VARIABLE_MAP.get(base_code, base_code)
            series = pd.to_numeric(group_df[col_name], errors='coerce').dropna()
            total  = len(series)
            if total == 0:
                continue

            counts = series.value_counts()
            for cat_code, count in counts.items():
                dist_rows.append({
                    'unit_id':  unit_id,
                    'la_name':  la_name,
                    'group':    gname,
                    'variable': label,
                    'category_code': cat_code,
                    'category': cat_map.get(cat_code, str(cat_code)),
                    'count':    int(count),
                    'pct':      round(count / total * 100, 1),
                })

    del df_unit
    gc.collect()

# ── Save ──────────────────────────────────────────────────────────────────────
df_baselines = pd.DataFrame(baseline_rows)
df_baselines.to_csv(BASELINES_CSV, index=False)
print(f"\nSaved {len(df_baselines)} baseline rows to {BASELINES_CSV.name}")

df_dist = pd.DataFrame(dist_rows)
df_dist.to_csv(DISTRIBUTIONS_CSV, index=False)
print(f"Saved {len(df_dist)} distribution rows to {DISTRIBUTIONS_CSV.name}")

display(df_baselines[['unit_id', 'group', 'size']].head(12))
display(df_dist.head(12))



🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  


🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  
  FOUR-LA SUBSET ACTIVE — Newham, Tower Hamlets, Islington, Hounslow only (4 LAs)
  Set USE_FOUR_LA_SUBSET = False for all London or full UK.
🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  🗺️  

Computing group averages for 4 units [USE_FOUR_LA_SUBSET=True]


Group averages: 100%|██████████| 4/4 [00:00<00:00,  5.92it/s]


Saved 24 baseline rows to group_baselines.csv
Saved 1850 distribution rows to group_distributions.csv


,unit_id,group,size
0,E09000018,Employed,84192
1,E09000018,Retired,22903
2,E09000018,Unemployed,8038
3,E09000018,Student,3142
4,E09000018,On leave,4720
5,E09000018,Inactive,5999
6,E09000019,Employed,84796
7,E09000019,Retired,17335
8,E09000019,Unemployed,8421
9,E09000019,Student,3119


,unit_id,la_name,group,variable,category_code,category,count,pct
0,E09000018,Hounslow,Employed,Sex (Derived),1.0,Male,46802,55.6
1,E09000018,Hounslow,Employed,Sex (Derived),2.0,Female,37390,44.4
2,E09000018,Hounslow,Employed,Ethnic group,1.0,White,48509,57.6
3,E09000018,Hounslow,Employed,Ethnic group,4.0,Pakistani / Bangladeshi,12934,15.4
4,E09000018,Hounslow,Employed,Ethnic group,3.0,Indian,11867,14.1
5,E09000018,Hounslow,Employed,Ethnic group,5.0,Other Asian,3114,3.7
6,E09000018,Hounslow,Employed,Ethnic group,2.0,Mixed,2319,2.8
7,E09000018,Hounslow,Employed,Ethnic group,8.0,African,1720,2.0
8,E09000018,Hounslow,Employed,Ethnic group,7.0,Caribbean,1560,1.9
9,E09000018,Hounslow,Employed,Ethnic group,6.0,Arab,1558,1.9
